# Binding and Stable Structures

**"From chaos to order: the emergence of particles."**

This notebook explores how stable structures (triads, atoms) emerge from the dynamics.

---

## Stable Configurations in FTD

| Structure | Description | Physical Analog |
|-----------|-------------|-----------------|
| Triad | 3 locked voxels in triangle | Nucleon (p, n) |
| Shell | Orbital arrangement | Electron orbitals |
| Cluster | Multiple triads | Atomic nucleus |

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.ndimage import label

repo_root = os.path.abspath("../../")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation, waves, forces, binding
from ternary_matrix.config import CONSTANTS

print("Modules loaded.")

## 1. The Triad: Fundamental Stable Unit

Three particles arranged in an equilateral triangle exhibit **enhanced stability**.

- Pairwise distance ≈ √2 lattice units
- Binding energy ≈ KB × φ (golden ratio)
- Decay is suppressed (locked state)

In [ ]:
# Create ideal triad geometry
def create_triad(universe, center, state=1):
    """Create a triad at the specified center."""
    cx, cy, cz = center
    
    # Equilateral triangle positions (approximately)
    # Using integer lattice positions that approximate equilateral
    positions = [
        (cx, cy + 1, cz),      # Top
        (cx - 1, cy, cz),      # Bottom-left
        (cx + 1, cy, cz),      # Bottom-right
    ]
    
    for pos in positions:
        universe.states[pos] = state
        universe.is_locked[pos] = True
        # Give flux support
        universe.flux[pos[0], pos[1], pos[2], :] = 2.0
    
    return positions

In [ ]:
# Visualize triad geometry
fig = plt.figure(figsize=(12, 5))

# 2D view
ax1 = fig.add_subplot(121)
triad_2d = np.array([
    [0, 1],      # Top
    [-1, 0],     # Bottom-left
    [1, 0],      # Bottom-right
])

# Draw triad
ax1.scatter(triad_2d[:, 0], triad_2d[:, 1], c='red', s=200, zorder=5)
for i in range(3):
    j = (i + 1) % 3
    ax1.plot([triad_2d[i, 0], triad_2d[j, 0]], 
             [triad_2d[i, 1], triad_2d[j, 1]], 'k-', linewidth=2)

# Labels
labels = ['u', 'u', 'd']  # Proton: uud
for i, (x, y) in enumerate(triad_2d):
    ax1.annotate(labels[i], (x, y), fontsize=14, ha='center', va='center', color='white')

ax1.set_xlim(-2, 2)
ax1.set_ylim(-1.5, 2)
ax1.set_aspect('equal')
ax1.set_title('Triad Geometry (Proton: uud)', fontsize=12)
ax1.grid(True, alpha=0.3)

# 3D view
ax2 = fig.add_subplot(122, projection='3d')
universe = Universe(size=16)
center = (8, 8, 8)
positions = create_triad(universe, center)

xs = [p[0] for p in positions]
ys = [p[1] for p in positions]
zs = [p[2] for p in positions]

ax2.scatter(xs, ys, zs, c='red', s=200)
for i in range(3):
    j = (i + 1) % 3
    ax2.plot([xs[i], xs[j]], [ys[i], ys[j]], [zs[i], zs[j]], 'k-', linewidth=2)

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('Triad in 3D Lattice', fontsize=12)

plt.tight_layout()
plt.show()

# Calculate distances
d01 = np.sqrt(sum((positions[0][i] - positions[1][i])**2 for i in range(3)))
d12 = np.sqrt(sum((positions[1][i] - positions[2][i])**2 for i in range(3)))
d20 = np.sqrt(sum((positions[2][i] - positions[0][i])**2 for i in range(3)))

print(f"\nPairwise distances: {d01:.2f}, {d12:.2f}, {d20:.2f}")
print(f"Target distance: √2 ≈ {np.sqrt(2):.2f}")

## 2. Stability Test: Triad vs Isolated Particles

Compare the lifetime of a triad vs isolated particles under decay.

In [ ]:
# Configure for stability test
CONSTANTS.C = 0.5
CONSTANTS.KB = 1.0
CONSTANTS.DECAY_RATE = 0.02  # Moderate decay
CONSTANTS.DAMPING = 0.01

# Test 1: Isolated particles
def test_isolated(n_particles=3):
    universe = Universe(size=32)
    center = universe.size // 2
    
    # Place particles far apart (no binding)
    for i in range(n_particles):
        pos = (center + i*5, center, center)
        universe.states[pos] = 1
        universe.flux[pos[0], pos[1], pos[2], :] = 2.0
    
    forces.calculate_density(universe)
    
    history = []
    for t in range(100):
        n_alive = np.count_nonzero(universe.states != 0)
        history.append(n_alive)
        master_equation.tick(universe)
    
    return history

# Test 2: Triad (locked)
def test_triad():
    universe = Universe(size=32)
    center = (16, 16, 16)
    positions = create_triad(universe, center)
    
    forces.calculate_density(universe)
    
    history = []
    for t in range(100):
        n_alive = np.count_nonzero(universe.states != 0)
        history.append(n_alive)
        master_equation.tick(universe)
    
    return history

isolated_history = test_isolated()
triad_history = test_triad()

In [ ]:
# Plot stability comparison
plt.figure(figsize=(10, 5))
plt.plot(isolated_history, 'b--', linewidth=2, label='Isolated Particles')
plt.plot(triad_history, 'r-', linewidth=2, label='Locked Triad')
plt.xlabel('Time (ticks)', fontsize=12)
plt.ylabel('Surviving Particles', fontsize=12)
plt.title('Stability: Triad vs Isolated Particles', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-0.5, 4)
plt.show()

print(f"Isolated: Decayed in {isolated_history.index(0) if 0 in isolated_history else '>100'} ticks")
print(f"Triad: {triad_history[-1]} particles remain after 100 ticks")

## 3. Cluster Detection: Finding Structures

Use connected component analysis to detect particle clusters.

In [ ]:
def detect_clusters(universe):
    """Detect connected clusters of manifested particles."""
    matter_mask = (universe.states != 0)
    labels, n_clusters = label(matter_mask)
    
    if n_clusters == 0:
        return []
    
    clusters = []
    for i in range(1, n_clusters + 1):
        cluster_mask = (labels == i)
        size = np.count_nonzero(cluster_mask)
        positions = np.argwhere(cluster_mask)
        center = positions.mean(axis=0)
        
        # Classify by size
        if size < 3:
            cluster_type = 'unstable'
        elif size <= 5:
            cluster_type = 'baryon'
        else:
            cluster_type = 'nucleus'
        
        clusters.append({
            'id': i,
            'size': size,
            'type': cluster_type,
            'center': center,
            'positions': positions
        })
    
    return clusters

In [ ]:
# Create a universe with multiple structures
CONSTANTS.KB = 1.0
CONSTANTS.DECAY_RATE = 0.001

universe = Universe(size=48)

# Create several triads at different locations
triad_centers = [
    (12, 24, 24),
    (24, 12, 24),
    (24, 36, 24),
    (36, 24, 24),
]

for center in triad_centers:
    create_triad(universe, center)

# Create some isolated particles
isolated_positions = [
    (10, 10, 24),
    (38, 38, 24),
]

for pos in isolated_positions:
    universe.states[pos] = 1
    universe.flux[pos[0], pos[1], pos[2], :] = 2.0

# Create a larger "nucleus" (cluster of triads)
nucleus_center = (24, 24, 24)
for dx in [-1, 0, 1]:
    for dy in [-1, 0, 1]:
        if abs(dx) + abs(dy) <= 1:
            pos = (nucleus_center[0]+dx, nucleus_center[1]+dy, nucleus_center[2])
            universe.states[pos] = 1 if (dx + dy) % 2 == 0 else -1
            universe.is_locked[pos] = True
            universe.flux[pos[0], pos[1], pos[2], :] = 3.0

forces.calculate_density(universe)

# Detect clusters
clusters = detect_clusters(universe)

print("Detected Structures:")
for c in clusters:
    print(f"  Cluster {c['id']}: size={c['size']}, type={c['type']}, center={c['center'].round(1)}")

In [ ]:
# Visualize detected structures
fig = plt.figure(figsize=(14, 5))

# 2D projection
ax1 = fig.add_subplot(121)
z_slice = 24

# Background
states_2d = universe.states[:, :, z_slice]
ax1.imshow(states_2d.T, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1, alpha=0.5)

# Mark clusters with different colors
colors = plt.cm.Set1(np.linspace(0, 1, len(clusters)))
for c, color in zip(clusters, colors):
    pos = c['positions']
    # Filter to z_slice
    mask = (pos[:, 2] == z_slice) | (abs(pos[:, 2] - z_slice) <= 1)
    if mask.any():
        ax1.scatter(pos[mask, 0], pos[mask, 1], c=[color], s=100, 
                   label=f"{c['type']} (size={c['size']})")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('Detected Clusters (z=24)', fontsize=12)
ax1.legend(loc='upper right', fontsize=8)

# 3D view
ax2 = fig.add_subplot(122, projection='3d')

for c, color in zip(clusters, colors):
    pos = c['positions']
    ax2.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=[color], s=50, 
               label=f"{c['type']} ({c['size']})")

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('3D Structure View', fontsize=12)
ax2.legend(loc='upper left', fontsize=8)

plt.suptitle('Structure Detection: Particles → Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Nucleosynthesis: Structure Formation Over Time

Watch structures form from an initial flux "soup".

In [ ]:
# Primordial soup simulation
CONSTANTS.C = 0.5
CONSTANTS.KB = 0.8
CONSTANTS.DECAY_RATE = 0.005
CONSTANTS.DAMPING = 0.02

universe = Universe(size=48)

# Random flux injection ("Big Bang")
universe.flux = np.random.normal(0, 2.0, universe.flux.shape).astype(np.float32)

forces.calculate_density(universe)
print(f"Initial conditions: max density = {universe.density.max():.2f}")

In [ ]:
# Track structure formation
structure_history = {
    't': [],
    'unstable': [],
    'baryon': [],
    'nucleus': [],
    'total_particles': [],
    'n_clusters': []
}

for t in range(100):
    clusters = detect_clusters(universe)
    
    n_unstable = sum(1 for c in clusters if c['type'] == 'unstable')
    n_baryon = sum(1 for c in clusters if c['type'] == 'baryon')
    n_nucleus = sum(1 for c in clusters if c['type'] == 'nucleus')
    n_total = np.count_nonzero(universe.states != 0)
    
    structure_history['t'].append(t)
    structure_history['unstable'].append(n_unstable)
    structure_history['baryon'].append(n_baryon)
    structure_history['nucleus'].append(n_nucleus)
    structure_history['total_particles'].append(n_total)
    structure_history['n_clusters'].append(len(clusters))
    
    master_equation.tick(universe)

print(f"Final state: {structure_history['total_particles'][-1]} particles in {structure_history['n_clusters'][-1]} clusters")

In [ ]:
# Visualize nucleosynthesis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Structure populations
ax = axes[0, 0]
ax.stackplot(structure_history['t'],
             structure_history['unstable'],
             structure_history['baryon'],
             structure_history['nucleus'],
             labels=['Unstable (<3)', 'Baryons (3-5)', 'Nuclei (>5)'],
             colors=['#ffcccc', '#ff6666', '#cc0000'])
ax.set_xlabel('Time (ticks)')
ax.set_ylabel('Number of Clusters')
ax.set_title('Structure Type Evolution')
ax.legend(loc='upper right')

# Total particles
ax = axes[0, 1]
ax.plot(structure_history['t'], structure_history['total_particles'], 'b-', linewidth=2)
ax.set_xlabel('Time (ticks)')
ax.set_ylabel('Total Manifested Particles')
ax.set_title('Particle Count Over Time')
ax.grid(True, alpha=0.3)

# Cluster count
ax = axes[1, 0]
ax.plot(structure_history['t'], structure_history['n_clusters'], 'g-', linewidth=2)
ax.set_xlabel('Time (ticks)')
ax.set_ylabel('Number of Clusters')
ax.set_title('Cluster Formation')
ax.grid(True, alpha=0.3)

# Final state visualization
ax = axes[1, 1]
z_slice = universe.size // 2
states_2d = universe.states[:, :, z_slice]
im = ax.imshow(states_2d.T, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Final State (z={z_slice})')
plt.colorbar(im, ax=ax, label='State')

plt.suptitle('Nucleosynthesis: From Soup to Structures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Binding Energy and the Golden Ratio

The binding energy of triads is related to **φ** (golden ratio ≈ 1.618).

In [ ]:
# Calculate binding energy
def calculate_binding_energy(universe, cluster):
    """Estimate binding energy from flux at particle positions."""
    positions = cluster['positions']
    total_flux = 0
    
    for pos in positions:
        flux_mag = np.linalg.norm(universe.flux[pos[0], pos[1], pos[2], :])
        total_flux += flux_mag
    
    return total_flux / len(positions)

# Golden ratio
phi = (1 + np.sqrt(5)) / 2
print(f"Golden Ratio φ = {phi:.6f}")
print(f"KB × φ = {CONSTANTS.KB * phi:.6f} (predicted binding energy scale)")

In [ ]:
# The Fibonacci sequence appears in cluster sizes
fib = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]

# Collect cluster size distribution from many simulations
all_sizes = []

for trial in range(20):
    universe = Universe(size=32)
    universe.flux = np.random.normal(0, 2.0, universe.flux.shape).astype(np.float32)
    
    for t in range(50):
        master_equation.tick(universe)
    
    clusters = detect_clusters(universe)
    for c in clusters:
        all_sizes.append(c['size'])

# Plot distribution
plt.figure(figsize=(10, 5))
bins = np.arange(0.5, 20.5, 1)
counts, _, _ = plt.hist(all_sizes, bins=bins, alpha=0.7, color='blue', edgecolor='black')

# Mark Fibonacci numbers
for f in fib[:7]:
    plt.axvline(f, color='red', linestyle='--', alpha=0.5)

plt.xlabel('Cluster Size', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Cluster Size Distribution (Red lines = Fibonacci numbers)', fontsize=14, fontweight='bold')
plt.xlim(0, 15)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print(f"\nMost common sizes: {sorted(set(all_sizes), key=all_sizes.count, reverse=True)[:5]}")

## 6. Shell Structures: Electron Orbitals

Negative particles can form shell-like structures around positive clusters.

In [ ]:
# Create a "hydrogen atom" analog
universe = Universe(size=32)
center = 16

# Nucleus (proton-like)
create_triad(universe, (center, center, center))

# Electron-like particle in orbit
# Place at radius ~ 3-4 (first Bohr radius analog)
universe.states[center + 4, center, center] = -1
universe.flux[center + 4, center, center, :] = 2.0

# Give orbital velocity
universe.wave_velocity[center + 4, center, center, 1] = 0.5  # y-velocity

forces.calculate_density(universe)

# Track electron position
electron_trajectory = []

for t in range(100):
    # Find electron position
    neg_positions = np.argwhere(universe.states == -1)
    if len(neg_positions) > 0:
        electron_trajectory.append(neg_positions[0].copy())
    
    master_equation.tick(universe)

In [ ]:
# Visualize orbit
if len(electron_trajectory) > 0:
    trajectory = np.array(electron_trajectory)
    
    fig = plt.figure(figsize=(12, 5))
    
    # 3D trajectory
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2], 'b-', alpha=0.5)
    ax1.scatter(trajectory[0, 0], trajectory[0, 1], trajectory[0, 2], c='green', s=100, label='Start')
    ax1.scatter(trajectory[-1, 0], trajectory[-1, 1], trajectory[-1, 2], c='red', s=100, label='End')
    
    # Nucleus
    ax1.scatter([center], [center], [center], c='orange', s=200, marker='*', label='Nucleus')
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Electron Orbit', fontsize=12)
    ax1.legend()
    
    # Radial distance over time
    ax2 = fig.add_subplot(122)
    radii = np.sqrt((trajectory[:, 0] - center)**2 + 
                    (trajectory[:, 1] - center)**2 + 
                    (trajectory[:, 2] - center)**2)
    ax2.plot(radii, 'b-', linewidth=2)
    ax2.axhline(4, color='gray', linestyle='--', label='Initial radius')
    ax2.set_xlabel('Time (ticks)')
    ax2.set_ylabel('Distance from Nucleus')
    ax2.set_title('Orbital Radius', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle('Shell Structure: Electron Orbiting Nucleus', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Electron decayed before tracking complete.")

## 7. Summary: Structure Formation

### Key Structures

| Structure | Size | Stability | Physical Analog |
|-----------|------|-----------|------------------|
| Isolated | 1-2 | Unstable | Quarks, mesons |
| Triad | 3 | Stable | Nucleon |
| Cluster | 3-5 | Stable | Light nuclei |
| Nucleus | >5 | Very stable | Heavy nuclei |

### Stability Mechanisms

1. **Locking** - Bound particles don't decay
2. **Geometry** - Triangle configuration minimizes energy
3. **Flux support** - Mutual flux reinforcement
4. **Golden ratio** - Binding energy ~ KB × φ

### Emergent Phenomena

- Nucleosynthesis from primordial soup
- Fibonacci clustering patterns
- Shell/orbital structures
- Matter-antimatter differentiation

**Next**: See `05_quantum_phenomena.ipynb` for interference and the Born rule.